# Notebook 01 — pandas Dasar sampai Groupby & Merge — KUNCI JAWABAN

**Dataset:** TokoKita (fiktif), toko online dengan 6 tabel: `customers`, `products`, `orders`, `order_items`, `marketing`, `web_events`.

**Cara memakai**
- **Google Colab:** File → Upload notebook → pilih file ini. Jalankan sel pertama, lalu upload 6 file CSV dari folder `latihan/data`.
- **Jupyter/VS Code di laptop:** buka langsung dari folder `latihan/python`; CSV dibaca otomatis dari `latihan/data`.

Kerjakan 15 soal berurutan. Soal-soal ini sama dengan Lab Python di website, sehingga jawabanmu bisa dicek di sana.
> Revenue baris = `qty × harga_satuan × (1 − diskon)`. Pendapatan hanya dihitung dari order berstatus **selesai**.

In [1]:
import os
import pandas as pd
import numpy as np

# ── Google Colab: jalankan sel ini, lalu upload 6 file CSV dari folder latihan/data ──
try:
    from google.colab import files  # noqa
    if not os.path.exists("orders.csv"):
        files.upload()
except ImportError:
    pass

DATA = "" if os.path.exists("orders.csv") else "../data/"   # di laptop: notebook ada di latihan/python, CSV di latihan/data
customers   = pd.read_csv(DATA + "customers.csv",   parse_dates=["tgl_daftar"])
products    = pd.read_csv(DATA + "products.csv")
orders      = pd.read_csv(DATA + "orders.csv",      parse_dates=["tgl_order"])
order_items = pd.read_csv(DATA + "order_items.csv")
marketing   = pd.read_csv(DATA + "marketing.csv",   parse_dates=["minggu"])
web_events  = pd.read_csv(DATA + "web_events.csv",  parse_dates=["tanggal"])
print({n: len(d) for n, d in [("customers", customers), ("products", products), ("orders", orders),
                              ("order_items", order_items), ("marketing", marketing), ("web_events", web_events)]})

{'customers': 600, 'products': 40, 'orders': 1332, 'order_items': 2217, 'marketing': 288, 'web_events': 5030}


### Soal 1. Jumlah order
DataFrame `orders` sudah tersedia. Berapa **jumlah baris**-nya?

*Petunjuk:* `len(orders) atau orders.shape[0]`

In [2]:
jawaban = len(orders)

jawaban

1332

### Soal 2. Produk olahraga
Buat DataFrame berisi kolom `nama_produk, harga` untuk kategori **Olahraga**, urut harga **termahal**.

*Petunjuk:* `products[products.kategori == "Olahraga"][["nama_produk", "harga"]].sort_values("harga", ascending=False)`

In [3]:
jawaban = products[products.kategori == "Olahraga"][["nama_produk", "harga"]].sort_values("harga", ascending=False)

jawaban

,nama_produk,harga
38,Sepatu Lari,599000
39,Tas Gym,219000
32,Matras Yoga,199000
33,Dumbbell 2kg (pasang),149000
37,Kaos Lari Dry-Fit,129000
36,Resistance Band,89000
35,Botol Shaker,69000
34,Tali Skipping,49000


### Soal 3. Proporsi status order
Hitung **proporsi** (0–1) setiap `status` di tabel orders.

*Petunjuk:* `orders["status"].value_counts(normalize=True)`

In [4]:
jawaban = orders["status"].value_counts(normalize=True)

jawaban

status
selesai    0.870871
batal      0.093093
retur      0.036036
Name: proportion, dtype: float64

### Soal 4. 5 produk termahal
Ambil kolom `nama_produk, harga` dari **5 produk termahal**.

*Petunjuk:* `sort_values("harga", ascending=False).head(5) atau nlargest(5, "harga")`

In [5]:
jawaban = products.nlargest(5, "harga")[["nama_produk", "harga"]]

jawaban

,nama_produk,harga
10,Smartwatch Lite,649000
38,Sepatu Lari,599000
14,Keyboard Mekanik,549000
7,Jaket Denim,399000
15,Webcam HD,399000


### Soal 5. Order selesai Oktober 2026
Berapa jumlah order berstatus **selesai** pada bulan **Oktober 2026**?

*Petunjuk:* `Filter 2 kondisi: (orders.status == "selesai") & (orders.tgl_order >= "2026-10-01")`

In [6]:
jawaban = len(orders[(orders.status == "selesai") & (orders.tgl_order >= "2026-10-01") & (orders.tgl_order < "2026-11-01")])

jawaban

87

### Soal 6. Kolom revenue
Tambahkan kolom `revenue` = qty × harga_satuan × (1 − diskon) ke `order_items`. `jawaban` = total revenue **semua** item (dibulatkan tanpa desimal).

*Petunjuk:* `round(order_items.revenue.sum())`

In [7]:
order_items["revenue"] = order_items.qty * order_items.harga_satuan * (1 - order_items.diskon)
jawaban = round(order_items.revenue.sum())

jawaban

560200500

### Soal 7. Order per bulan 2026
Buat kolom `bulan` (format 'YYYY-MM') di orders, lalu hitung **jumlah order per bulan** untuk tahun 2026 (semua status), urut bulan.

*Petunjuk:* `orders.tgl_order.dt.strftime("%Y-%m"); filter dt.year == 2026; groupby("bulan").size()`

In [8]:
orders["bulan"] = orders.tgl_order.dt.strftime("%Y-%m")
jawaban = orders[orders.tgl_order.dt.year == 2026].groupby("bulan").size()

jawaban

bulan
2026-01     80
2026-02     62
2026-03     98
2026-04     81
2026-05     82
2026-06     85
2026-07     85
2026-08     89
2026-09     75
2026-10    102
dtype: int64

### Soal 8. Segmen harga
Dengan `pd.cut`, buat segmen harga produk: Murah (≤100.000), Sedang (≤300.000), Premium (sisanya). `jawaban` = jumlah produk per segmen.

*Petunjuk:* `pd.cut(products.harga, bins=[0, 100000, 300000, float("inf")], labels=["Murah","Sedang","Premium"]) lalu value_counts()`

In [9]:
products["segmen"] = pd.cut(products.harga, bins=[0, 100000, 300000, float("inf")], labels=["Murah", "Sedang", "Premium"])
jawaban = products.segmen.value_counts()

jawaban

segmen
Sedang     20
Murah      14
Premium     6
Name: count, dtype: int64

### Soal 9. Order akhir pekan
Berapa order (semua status) yang dibuat di hari **Sabtu atau Minggu**?

*Petunjuk:* `orders.tgl_order.dt.dayofweek >= 5 (Senin=0)`

In [10]:
jawaban = int((orders.tgl_order.dt.dayofweek >= 5).sum())

jawaban

398

### Soal 10. Revenue per kategori
Hitung revenue **order selesai** per `kategori` (Series, dibulatkan tanpa desimal), urut terbesar. Kamu perlu menggabungkan order_items, orders, products.

*Petunjuk:* `merge(orders, on="order_id").merge(products, on="product_id"); filter status; groupby("kategori")["revenue"].sum().round().sort_values(ascending=False)`

In [11]:
df = order_items.merge(orders, on="order_id").merge(products, on="product_id")
df["revenue"] = df.qty * df.harga_satuan * (1 - df.diskon)
jawaban = df[df.status == "selesai"].groupby("kategori")["revenue"].sum().round().sort_values(ascending=False)

jawaban

kategori
Elektronik      145574900.0
Rumah Tangga    119210400.0
Fashion          91841100.0
Olahraga         85639600.0
Kecantikan       40612400.0
Name: revenue, dtype: float64

### Soal 11. Crosstab kota × channel
Buat tabel frekuensi jumlah pelanggan dengan **kota** sebagai baris dan **channel_akuisisi** sebagai kolom.

*Petunjuk:* `pd.crosstab(customers.kota, customers.channel_akuisisi)`

In [12]:
jawaban = pd.crosstab(customers.kota, customers.channel_akuisisi)

jawaban

channel_akuisisi,Google Ads,Instagram,Organik,Referral,TikTok
kota,,,,,
Balikpapan,1,8,7,6,6
Bandung,17,11,20,9,17
Denpasar,8,11,3,3,6
Jakarta,25,33,42,28,38
Makassar,13,10,13,2,4
Medan,5,14,9,10,9
Palembang,2,11,10,6,8
Semarang,8,8,11,4,9
Surabaya,21,15,25,8,19


### Soal 12. Top 5 pelanggan
DataFrame `customer_id, nama, total` untuk **5 pelanggan** dengan total belanja order selesai terbesar (total dibulatkan), urut terbesar.

*Petunjuk:* `Hitung revenue per customer_id, nlargest(5), merge ke customers untuk nama`

In [13]:
df = order_items.merge(orders, on="order_id")
df = df[df.status == "selesai"]
df["revenue"] = df.qty * df.harga_satuan * (1 - df.diskon)
t = df.groupby("customer_id")["revenue"].sum().nlargest(5).round().rename("total").reset_index()
jawaban = t.merge(customers[["customer_id", "nama"]], on="customer_id")[["customer_id", "nama", "total"]]

jawaban

,customer_id,nama,total
0,C0309,Umi Anggraini,3868900.0
1,C0009,Vina Permata,3806400.0
2,C0327,Intan Anggraini,3597700.0
3,C0110,Bella Lestari,3551700.0
4,C0220,Wahyu Halim,3549500.0


### Soal 13. AOV
Hitung **AOV** (rata-rata nilai per order) untuk order selesai, dibulatkan tanpa desimal.

*Petunjuk:* `Total revenue per order_id dulu (groupby), lalu mean()`

In [14]:
df = order_items.merge(orders, on="order_id")
df = df[df.status == "selesai"]
df["revenue"] = df.qty * df.harga_satuan * (1 - df.diskon)
jawaban = round(df.groupby("order_id")["revenue"].sum().mean())

jawaban

416274

### Soal 14. Pivot bulan × kategori
Buat **pivot_table** revenue order selesai tahun **2026**: index = bulan ('YYYY-MM'), columns = kategori, nilai = sum revenue (dibulatkan), isi kosong dengan 0.

*Petunjuk:* `df.pivot_table(index="bulan", columns="kategori", values="revenue", aggfunc="sum", fill_value=0).round()`

In [15]:
df = order_items.merge(orders, on="order_id").merge(products, on="product_id")
df = df[(df.status == "selesai") & (df.tgl_order.dt.year == 2026)].copy()
df["revenue"] = df.qty * df.harga_satuan * (1 - df.diskon)
df["bulan"] = df.tgl_order.dt.strftime("%Y-%m")
jawaban = df.pivot_table(index="bulan", columns="kategori", values="revenue", aggfunc="sum", fill_value=0).round()

jawaban

kategori,Elektronik,Fashion,Kecantikan,Olahraga,Rumah Tangga
bulan,,,,,
2026-01,10433400.0,6440000.0,2642700.0,4392900.0,7704000.0
2026-02,4726000.0,7101900.0,1272200.0,5005300.0,4581000.0
2026-03,11528500.0,8244400.0,2931600.0,7299400.0,7474300.0
2026-04,6565600.0,5176400.0,2954400.0,5163300.0,8951800.0
2026-05,10440500.0,3317500.0,2313500.0,5952900.0,6243000.0
2026-06,10078200.0,4818100.0,2807200.0,3515000.0,10736800.0
2026-07,9710900.0,3037100.0,3298700.0,3762500.0,5710600.0
2026-08,6891200.0,7402800.0,2889800.0,5751000.0,8656200.0
2026-09,7743500.0,3246100.0,2205200.0,6557800.0,7254400.0


### Soal 15. Repeat rate
Dari pelanggan yang punya order selesai, berapa **persen** yang punya ≥ 2 order selesai? (2 desimal)

*Petunjuk:* `n = orders[...].groupby('customer_id').size(); round((n >= 2).mean() * 100, 2)`

In [16]:
n = orders[orders.status == "selesai"].groupby("customer_id").size()
jawaban = round((n >= 2).mean() * 100, 2)

jawaban

np.float64(52.84)

---
Bandingkan dengan jawabanmu. Ingat: ada banyak cara benar untuk menulis kode yang sama.